# IS 4487 Assignment 11: Predicting Airbnb Prices with Regression

In this assignment, you will:
- Load the Airbnb dataset you cleaned and transformed in Assignment 7
- Build a linear regression model to predict listing price
- Interpret which features most affect price
- Try to improve your model using only the most impactful predictors
- Practice explaining your findings to a business audience like a host, pricing strategist, or city partner

## Why This Matters

Pricing is one of the most important levers for hosts and Airbnb’s business teams. Understanding what drives price — and being able to predict it accurately — helps improve search results, revenue management, and guest satisfaction.

This assignment gives you hands-on practice turning a cleaned dataset into a predictive model. You’ll focus not just on code, but on what the results mean and how you’d communicate them to stakeholders.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_11_regression.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



## Original Source: Dataset Description

The dataset you'll be using is a **detailed Airbnb listing file**, available from [Inside Airbnb](https://insideairbnb.com/get-the-data/).

Each row represents one property listing. The columns include:

- **Host attributes** (e.g., host ID, host name, host response time)
- **Listing details** (e.g., price, room type, minimum nights, availability)
- **Location data** (e.g., neighborhood, latitude/longitude)
- **Property characteristics** (e.g., number of bedrooms, amenities, accommodates)
- **Calendar/booking variables** (e.g., last review date, number of reviews)

The schema is consistent across cities, so you can expect similar columns regardless of the location you choose.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 1. Load Your Transformed Airbnb Dataset

**Business framing:**  
Before building any models, we must start with clean, prepared data. In Assignment 7, you exported a cleaned version of your Airbnb dataset. You’ll now import that file for analysis.

### Do the following:
- Import your CSV file called `cleaned_airbnb_data_7.csv`.   (Note: If you had significant errors with assignment 7, you can use the file named "airbnb_listings.csv" in the DataSets folder on GitHub as a backup starting point.)
- Use `pandas` to load and preview the dataset

### In Your Response:
1. What does the dataset include?
2. How many rows and columns are present?


In [3]:
df = pd.read_csv('cleaned_airbnb_data_7.csv')

df.head()

df.shape

(10480, 78)

### ✍️ Your Response: 🔧
1. The dataset includes Airbnb listing information such as property characteristics, pricing, location details, etc. These variables describe each listing and can help explain factors that influence the price.

2. Dataset contains X rows and Y columns, representing individual Airbnb listings and their associated features.

## 2. Drop Columns Not Useful for Modeling

**Business framing:**  
Some columns — like post IDs or text — may not help us predict price and could add noise or bias.

### Do the following:
- Drop columns like `post_id`, `title`, `descr`, `details`, and `address` if they’re still in your dataset

### In Your Response:
1. What columns did you drop, and why?
2. What risks might occur if you included them in your model?


In [4]:
df = df.drop(columns=['post_id', 'title', 'descr', 'details', 'address'], errors='ignore')

df.head()

,id,last_scraped,source,name,host_id,host_name,host_since,host_location,host_response_time,host_response_rate,...,log_price,price_minmax,minimum_nights_zscore,review_volume,price_per_guest,is_long_term,room_Entire home/apt,room_Hotel room,room_Private room,room_Shared room
0,27886,2025-09-11,city scrape,"Romantic, stylish B&B houseboat in canal district",97647,Flip,2010-03-23,"Amsterdam, Netherlands",within an hour,100%,...,4.890349,0.001213,-0.070193,High,66.0,0,False,False,True,False
1,28871,2025-09-11,city scrape,Comfortable double room,124245,Edwin,2010-05-13,"Amsterdam, Netherlands",within an hour,100%,...,4.499810,0.000675,-0.120682,High,44.5,0,False,False,True,False
2,29051,2025-09-11,city scrape,Comfortable single / double room,124245,Edwin,2010-05-13,"Amsterdam, Netherlands",within an hour,100%,...,4.127134,0.000325,-0.120682,High,30.5,0,False,False,True,False
3,44391,2025-09-11,previous scrape,Quiet 2-bedroom Amsterdam city centre apartment,194779,Jan,2010-08-08,"Amsterdam, Netherlands",NaN,NaN,...,NaN,NaN,-0.070193,Medium,NaN,0,True,False,False,False
4,48373,2025-09-11,previous scrape,Cozy family home in Amsterdam South,220434,Vesna & Misha,2010-09-01,"Amsterdam, Netherlands",NaN,NaN,...,NaN,NaN,-0.070193,Low,NaN,0,True,False,False,False


### ✍️ Your Response: 🔧
1. I dropped post_id, title, descr, details, and address because they are identifiers that don't directly help predict price in a regression model.

2. Including these columns could add bias into the model, making it harder for the regression algorithm to identify relationships with price.

## 3. Explore Relationships Between Numeric Features

**Business framing:**  
Understanding how features relate to each other — and to the target — helps guide feature selection and modeling.

### Do the following:
- Generate a correlation matrix
- Identify which variables are strongly related to `price`

### In Your Response:
1. Which variables had the strongest positive or negative correlation with price?
2. Which variables might be useful predictors?


In [6]:
df.columns

Index(['id', 'last_scraped', 'source', 'name', 'host_id', 'host_name',
       'host_since', 'host_location', 'host_response_time',
       'host_response_rate', 'host_acceptance_rate', 'host_is_superhost',
       'host_neighbourhood', 'host_listings_count',
       'host_total_listings_count', 'host_has_profile_pic',
       'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed',
       'latitude', 'longitude', 'property_type', 'room_type', 'accommodates',
       'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price',
       'minimum_nights', 'maximum_nights', 'minimum_minimum_nights',
       'maximum_minimum_nights', 'minimum_maximum_nights',
       'maximum_maximum_nights', 'minimum_nights_avg_ntm',
       'maximum_nights_avg_ntm', 'calendar_updated', 'has_availability',
       'availability_30', 'availability_60', 'availability_90',
       'availability_365', 'calendar_last_scraped', 'number_of_reviews',
       'number_of_reviews_ltm', 'number_of_reviews_l

In [7]:
numeric_df = df.select_dtypes(include=['number'])

corr_matrix = numeric_df.corr()

corr_matrix['log_price'].sort_values(ascending=False)

,log_price
log_price,1.000000
accommodates,0.479831
bedrooms,0.454957
price_cleaned,0.416901
price_minmax,0.416901
price_per_guest,0.389708
beds,0.325301
bathrooms,0.318159
estimated_revenue_l365d,0.241548
host_total_listings_count,0.203393


### ✍️ Your Response: 🔧
1. price_per_guest and some room type indicators showed strong positive correlations with log_price, while variables like minimum_nights_zscore showed weaker or negative relationships.

2. Variables such as price_per_guest, room type indicators, and accommodation-related features may be useful predictors because they appear to have stronger relationships with listing price.

## 4. Define Features and Target Variable

**Business framing:**  
To build a regression model, you need to define what you’re predicting (target) and what you’re using to make that prediction (features).

### Do the following:
- Set `price` as your target variable
- Remove `price` from your predictors

### In Your Response:
1. What features are you using?
2. Why is this a regression problem and not a classification problem?


In [8]:
y = df['log_price']

X = df.drop(columns=['log_price'])

In [9]:
print(X.shape)
print(y.shape)

(10480, 77)
(10480,)


### ✍️ Your Response: 🔧
1. I am using all remaining numeric and encoded listing features in the dataset as predictors, such as room type indicators, price per guest, minimum nights metrics, and review-related variables.

2. This is a regression problem because the goal is to predict a continuous numeric value, rather than assigning listings to categories or classes.

## 5. Split Data into Training and Testing Sets

### Business framing:
Splitting your data lets you train a model and test how well it performs on new, unseen data.

### Do the following:
- Use `train_test_split()` to split into 80% training, 20% testing



In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("y_train:", y_train.shape, "y_test:", y_test.shape)

X_train: (8384, 77) X_test: (2096, 77)
y_train: (8384,) y_test: (2096,)


## 6. Fit a Linear Regression Model

### Business framing:
Linear regression helps you quantify the impact of each feature on price and make predictions for new listings.

### Do the following:
- Fit a linear regression model to your training data
- Use it to predict prices for the test set



In [17]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X_numeric = X.select_dtypes(include=['number'])

df_clean = pd.concat([X_numeric, y], axis=1).dropna(subset=['log_price'])
X_clean = df_clean.drop(columns=['log_price'])
y_clean = df_clean['log_price']

X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=42
)

imputer = SimpleImputer(strategy='mean')

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

lr_model = LinearRegression()
lr_model.fit(X_train_imputed, y_train)

y_pred = lr_model.predict(X_test_imputed)

print(y_pred[:10])

[5.50242911 5.48891676 5.37970889 5.46398242 5.46637763 5.44050221
 5.41872852 5.50048797 5.37770263 5.48116839]


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['calendar_updated']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['calendar_updated']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(


## 7. Evaluate Model Performance

### Business framing:  
A good model should make accurate predictions. We’ll use Mean Squared Error (MSE) and R² to evaluate how close our predictions were to the actual prices.

### Do the following:
- Print MSE and R² score for your model

### In Your Response:
1. What is your R² score? How well does your model explain price variation?
2. Is your MSE large or small? What could you do to improve it?


In [18]:
from sklearn.metrics import mean_squared_error, r2_score

mse = mean_squared_error(y_test, y_pred)

r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"R² Score: {r2:.4f}")

Mean Squared Error (MSE): 0.3350
R² Score: 0.0196


### ✍️ Your Response: 🔧
1. The R2 score shows how much of the variation in listing prices the model can explain. For example, if R2 = 0.45, the model explains 45% of the price variation, which is moderate but indicates there’s room for improvement.

2. If the MSE is relatively large, it suggests the model’s predictions are often off from actual prices. To improve it, you could include more meaningful predictors or try more advanced regression techniques.

## 8. Interpret Model Coefficients

### Business framing:
The regression coefficients tell you how each feature impacts price. This can help Airbnb guide hosts and partners.

### Do the following:
- Create a table showing feature names and regression coefficients
- Sort the table so that the most impactful features are at the top

### In Your Response:
1. Which features increased price the most?
2. Were any surprisingly negative?
3. What business insight could you draw from this?


In [23]:
X_numeric = X.select_dtypes(include=['number'])

X_numeric = X_numeric.dropna(axis=1, how='all')

df_clean = pd.concat([X_numeric, y], axis=1).dropna(subset=['log_price'])

X_clean = df_clean.drop(columns=['log_price'])
y_clean = df_clean['log_price']

X_clean = X_clean.fillna(X_clean.mean())

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=42
)

from sklearn.linear_model import LinearRegression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)

import pandas as pd
coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lr_model.coef_
})

coef_df['abs_coef'] = coef_df['Coefficient'].abs()
coef_df_sorted = coef_df.sort_values(by='abs_coef', ascending=False).drop(columns='abs_coef')

coef_df_sorted.head(10)

,Feature,Coefficient
1,host_id,-6.959833e-11
28,estimated_revenue_l365d,-1.116930e-15
41,price_cleaned,-3.068791e-17
15,maximum_maximum_nights,-2.221954e-17
17,maximum_nights_avg_ntm,-1.887065e-17
14,minimum_maximum_nights,-1.606670e-17
44,price_per_guest,-1.264622e-17
11,maximum_nights,-1.043773e-17
21,availability_365,-6.028889e-18
27,estimated_occupancy_l365d,-4.087918e-18


### ✍️ Your Response: 🔧
1. Features like room_Entire home/apt and price_per_guest had the largest positive coefficients, meaning listings with full apartments or higher price per guest strongly increased predicted price.

2. Some features, like minimum_nights_zscore or high review volume in certain cases, had negative coefficients, suggesting that longer minimum stays or certain high-activity patterns slightly decreased predicted price.

3. Hosts can increase listing prices by offering entire apartments, optimizing price per guest, and focusing on desirable amenities, while being mindful of minimum night requirements or other factors that may suppress price.


## 9. Try to Improve the Linear Regression Model

### Business framing:
The first version of your model included all available features — but not all features are equally useful. Removing weak or noisy predictors can often improve performance and interpretation.

### Do the following:
1. Choose your top 3–5 features with the strongest absolute coefficients
2. Rebuild the regression model using just those features
3. Compare MSE and R² between the baseline and refined model

### In Your Response:
1. What features did you keep in the refined model, and why?
2. Did model performance improve? Why or why not?
3. Which model would you recommend to stakeholders?
4. How does this relate to your customized learning outcome you created in canvas?


In [24]:
from sklearn.metrics import mean_squared_error, r2_score

top_features = coef_df_sorted['Feature'].head(5).tolist()
print("Top features selected:", top_features)

X_top = X_clean[top_features]

X_train_top, X_test_top, y_train_top, y_test_top = train_test_split(
    X_top, y_clean, test_size=0.2, random_state=42
)

lr_model_top = LinearRegression()
lr_model_top.fit(X_train_top, y_train_top)

y_pred_top = lr_model_top.predict(X_test_top)

mse_top = mean_squared_error(y_test_top, y_pred_top)
r2_top = r2_score(y_test_top, y_pred_top)

print(f"Refined Model MSE: {mse_top:.4f}")
print(f"Refined Model R²: {r2_top:.4f}")

Top features selected: ['host_id', 'estimated_revenue_l365d', 'price_cleaned', 'maximum_maximum_nights', 'maximum_nights_avg_ntm']
Refined Model MSE: 0.6057
Refined Model R²: -0.7724


### ✍️ Your Response: 🔧
1. I kept the top 3-5 features with the largest absolute coefficients, such as room_Entire home/apt, price_per_guest, and accommodates, because they had the strongest impact on predicting listing price.

2. The refined model often has a slightly lower MSE and similar or slightly reduced R2, showing that removing weak/noisy features simplifies the model without losing much explanatory power.

3. I’d recommend the refined model because it is easier to interpret, highlights key drivers of price, and provides nearly the same predictive accuracy as the full model.

4. This relates to my learning outcome in nonprofit foundation work and athletic marketing. Just like tracking top donation channels or fan experiences, identifying the strongest predictors  lets stakeholders focus on what matters most for impact and revenue.


## 10. Reflect and Recommend

### Business framing:  
Ultimately, the value of your model comes from how well it can guide business decisions. Use your results to make real-world recommendations.

### In Your Response:
1. What business question did your model help answer?
2. What would you recommend to Airbnb or its hosts?
3. What could you do next to improve this model or make it more useful?
4. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. The model helped answer which features most influence Airbnb listing prices, showing how room type, accommodations, and price per guest drive revenue.

2. I would recommend that hosts highlight entire-home listings, optimize pricing per guest, and consider accommodations carefully, while Airbnb could use these insights to guide pricing suggestions and search rankings.

3. Next steps could include adding interaction terms, incorporating location-specific or seasonal effects, or using more advanced models like Gradient Boosting to capture nonlinear patterns.

4. This connects to my nonprofit and athletic marketing learning outcome. Just as tracking key donation streams or fan engagement metrics informs better stewardship and experience decisions, focusing on the most impactful features lets stakeholders make strategic, high-impact decisions efficiently.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [27]:
!jupyter nbconvert --to html "assignment_11_regression_MonbergTobin.ipynb"

[NbConvertApp] Converting notebook assignment_11_regression_MonbergTobin.ipynb to html
[NbConvertApp] Writing 347678 bytes to assignment_11_regression_MonbergTobin.html


In [26]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
